In [0]:
%run ./00_config

In [0]:
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("zone", "")
dbutils.widgets.text("run_id", "")
START_DATE = dbutils.widgets.get("start_date").strip()
END_DATE = dbutils.widgets.get("end_date").strip()
ZONE = dbutils.widgets.get("zone").strip().upper()
RUN_ID = dbutils.widgets.get("run_id").strip()
print("START_DATE:", START_DATE, "END_DATE:", END_DATE, "ZONE:", ZONE, "RUN_ID:", RUN_ID)

In [0]:
# Databricks notebook source
from pyspark.sql import functions as F
def parse_mtu_start(col_name: str):
    return F.expr(f"""
      coalesce(
        try_to_timestamp(trim(regexp_extract({col_name}, '^(.*?) - ', 1)), 'dd/MM/yyyy HH:mm:ss'),
        try_to_timestamp(trim(regexp_extract({col_name}, '^(.*?) - ', 1)), 'dd/MM/yyyy HH:mm')
      )
    """)
def normalize_zone(col_name: str):
    return F.regexp_replace(F.col(col_name), r"BZN\|", "")

In [0]:
# UNION VIEWS
prices_raw = spark.sql(f"SELECT * FROM {T_PRICES_2020_RAW} UNION ALL SELECT * FROM {T_PRICES_2021_RAW}")
load_raw = spark.sql(f"SELECT * FROM {T_LOAD_2020_RAW} UNION ALL SELECT * FROM {T_LOAD_2021_RAW}")
gen_raw = spark.sql(f"SELECT * FROM {T_GEN_2020_RAW} UNION ALL SELECT * FROM {T_GEN_2021_RAW}")
weather_raw = spark.sql(f"SELECT * FROM {T_WEATHER_2020_RAW} UNION ALL SELECT * FROM {T_WEATHER_2021_RAW}")
stations_raw = spark.table(T_STATIONS_RAW)

In [0]:
# PRICES CLEAN
prices_clean = (
    prices_raw
    .withColumn("event_time_local", parse_mtu_start("mtu_cet_cest"))
    .withColumn("zone", normalize_zone("area"))
    .withColumn("price_eur_mwh", F.expr("try_cast(dayahead_price_eur_mwh as double)"))
    .filter(F.col("event_time_local").isNotNull() & F.col("price_eur_mwh").isNotNull())
    .select(
        F.to_utc_timestamp("event_time_local", TIMEZONE).alias("event_time_utc"),
        "zone", "price_eur_mwh"
    )
    .dropDuplicates(["event_time_utc", "zone"])
)

In [0]:
# PASTE FILTER HERE
prices_clean = prices_clean.filter(
    (F.col("event_time_utc") >= F.to_timestamp(F.lit(START_DATE))) &
    (F.col("event_time_utc") < F.to_timestamp(F.lit(END_DATE)))
)
if ZONE:
    prices_clean = prices_clean.filter(F.upper(F.col("zone")) == ZONE.upper())
# then write
prices_clean.write.mode("overwrite").format("delta").saveAsTable(T_PRICES_CLEAN)


In [0]:

# LOAD CLEAN (qh -> hour avg)
load_clean = (
    load_raw
    .withColumn("event_time_local", parse_mtu_start("mtu_cet_cest"))
    .withColumn("zone", normalize_zone("area"))
    .withColumn("actual_total_load_mw", F.expr("try_cast(actual_load_mw as double)"))
    .withColumn("day_ahead_total_load_forecast_mw", F.expr("try_cast(forecast_load_mw as double)"))
    .filter(F.col("event_time_local").isNotNull())
    .withColumn("event_hour_local", F.date_trunc("hour", F.col("event_time_local")))
    .groupBy("event_hour_local", "zone")
    .agg(
        F.avg("actual_total_load_mw").alias("avg_actual_total_load_mw"),
        F.avg("day_ahead_total_load_forecast_mw").alias("day_ahead_total_load_forecast_mw")
    )
    .select(
        F.to_utc_timestamp("event_hour_local", TIMEZONE).alias("event_time_utc"),
        "zone",
        "avg_actual_total_load_mw",
        "day_ahead_total_load_forecast_mw"
    )
    .dropDuplicates(["event_time_utc", "zone"])
)

In [0]:
load_clean = load_clean.filter(
    (F.col("event_time_utc") >= F.to_timestamp(F.lit(START_DATE))) &
    (F.col("event_time_utc") < F.to_timestamp(F.lit(END_DATE)))
)
if ZONE:
    load_clean = load_clean.filter(F.upper(F.col("zone")) == ZONE.upper())
load_clean.write.mode("overwrite").format("delta").saveAsTable(T_LOAD_CLEAN)

In [0]:

# GENERATION BY TYPE + WIDE
gen_by_type = (
    gen_raw
    .withColumn("event_time_local", parse_mtu_start("mtu_cet_cest"))
    .withColumn("zone", normalize_zone("area"))
    .withColumn("production_type", F.col("production_type"))
    .withColumn("generation_mw", F.expr("try_cast(generation_mw as double)"))
    .filter(F.col("event_time_local").isNotNull() & F.col("generation_mw").isNotNull())
    .withColumn("event_hour_local", F.date_trunc("hour", F.col("event_time_local")))
    .withColumn("event_time_utc", F.to_utc_timestamp("event_hour_local", TIMEZONE))
    .groupBy("event_time_utc", "zone", "production_type")
    .agg(F.avg("generation_mw").alias("generation_mw_avg"))
)
gen_by_type.write.mode("overwrite").format("delta").saveAsTable(T_GEN_BY_TYPE)
base_for_pivot = (
    gen_by_type
    .withColumn("type_col", F.lower(F.col("production_type")))
    .withColumn("type_col", F.regexp_replace("type_col", r"[^a-z0-9]+", "_"))
    .withColumn("type_col", F.regexp_replace("type_col", r"_+", "_"))
    .withColumn("type_col", F.regexp_replace("type_col", r"^_|_$", ""))
    .select("event_time_utc", "zone", "type_col", "generation_mw_avg")
)
type_cols = [r["type_col"] for r in base_for_pivot.select("type_col").distinct().collect() if r["type_col"]]
gen_wide = (
    base_for_pivot
    .groupBy("event_time_utc", "zone")
    .pivot("type_col", type_cols)
    .agg(F.first("generation_mw_avg"))
    .fillna(0.0)
)
if type_cols:
    gen_wide = gen_wide.withColumn(
        "total_generation_mw_avg",
        F.expr(" + ".join([f"`{c}`" for c in type_cols]))
    )
else:
    gen_wide = gen_wide.withColumn("total_generation_mw_avg", F.lit(0.0))

gen_wide = gen_wide.filter(
    (F.col("event_time_utc") >= F.to_timestamp(F.lit(START_DATE))) &
    (F.col("event_time_utc") < F.to_timestamp(F.lit(END_DATE)))
)
if ZONE:
    gen_wide = gen_wide.filter(F.upper(F.col("zone")) == ZONE.upper())
gen_wide.write.mode("overwrite").format("delta").saveAsTable(T_GEN_WIDE)


In [0]:

# WEATHER CLEAN
weather_clean = (
    weather_raw
    .withColumn("event_time_utc", F.to_timestamp("event_time_utc"))
    .withColumn("zone", F.upper(F.col("zone")))
    .withColumn("temperature_c", F.expr("try_cast(temperature_c as double)"))
    .withColumn("u10_ms", F.expr("try_cast(u10_ms as double)"))
    .withColumn("v10_ms", F.expr("try_cast(v10_ms as double)"))
    .withColumn("surface_pressure_pa", F.expr("try_cast(surface_pressure_pa as double)"))
    .withColumn("wind_speed_ms", F.expr("sqrt(u10_ms*u10_ms + v10_ms*v10_ms)"))
    .filter(F.col("event_time_utc").isNotNull())
    .select("event_time_utc", "zone", "temperature_c", "u10_ms", "v10_ms", "wind_speed_ms", "surface_pressure_pa")
)

weather_clean = weather_clean.filter(
    (F.col("event_time_utc") >= F.to_timestamp(F.lit(START_DATE))) &
    (F.col("event_time_utc") < F.to_timestamp(F.lit(END_DATE)))
)
if ZONE:
    weather_clean = weather_clean.filter(F.upper(F.col("zone")) == ZONE.upper())
weather_clean.write.mode("overwrite").format("delta").saveAsTable(T_WEATHER_CLEAN)

In [0]:
# STATIONS CLEAN
stations_clean = (
    stations_raw
    .withColumn("id", F.expr("try_cast(id as int)"))
    .withColumn("country", F.upper(F.col("country")))
    .withColumn("status_id", F.expr("try_cast(status_id as int)"))
    .withColumn("latitude", F.expr("try_cast(latitude as double)"))
    .withColumn("longitude", F.expr("try_cast(longitude as double)"))
    .withColumn("snapshot_ts_utc", F.to_timestamp("snapshot_ts_utc"))
    .filter((F.col("country") == "NL") & F.col("latitude").between(-90, 90) & F.col("longitude").between(-180, 180))
    .select("id", "station_name", "country", "status_id", "status_name", "latitude", "longitude", "snapshot_ts_utc")
)

if ZONE:
    stations_clean = stations_clean.filter(F.upper(F.col("country")) == ZONE.upper())
stations_clean.write.mode("overwrite").format("delta").saveAsTable(T_STATIONS_CLEAN)

In [0]:
from pyspark.sql import functions as F
prices = spark.table(T_PRICES_CLEAN).filter((F.col("event_time_utc") >= F.to_timestamp(F.lit(START_DATE))) & (F.col("event_time_utc") < F.to_timestamp(F.lit(END_DATE))))
load = spark.table(T_LOAD_CLEAN).filter((F.col("event_time_utc") >= F.to_timestamp(F.lit(START_DATE))) & (F.col("event_time_utc") < F.to_timestamp(F.lit(END_DATE))))
gen = spark.table(T_GEN_WIDE).filter((F.col("event_time_utc") >= F.to_timestamp(F.lit(START_DATE))) & (F.col("event_time_utc") < F.to_timestamp(F.lit(END_DATE))))
weather = spark.table(T_WEATHER_CLEAN).filter((F.col("event_time_utc") >= F.to_timestamp(F.lit(START_DATE))) & (F.col("event_time_utc") < F.to_timestamp(F.lit(END_DATE))))
training_base = (
    prices.alias("p")
    .join(load.alias("l"), ["event_time_utc", "zone"], "inner")
    .join(gen.alias("g"), ["event_time_utc", "zone"], "left")
    .join(weather.alias("w"), ["event_time_utc", "zone"], "left")
    .withColumn("hour_of_day", F.hour("event_time_utc"))
    .withColumn("day_of_week", F.dayofweek("event_time_utc"))
    .withColumn("month", F.month("event_time_utc"))
)

training_base = training_base.filter(
    (F.col("event_time_utc") >= F.to_timestamp(F.lit(START_DATE))) &
    (F.col("event_time_utc") < F.to_timestamp(F.lit(END_DATE)))
)
if ZONE:
    training_base = training_base.filter(F.upper(F.col("zone")) == ZONE.upper())
training_base.write.mode("overwrite").format("delta").saveAsTable(T_TRAINING_BASE)


In [0]:
training_base.selectExpr("count(*) as rows", "min(event_time_utc) as min_ts", "max(event_time_utc) as max_ts").show()
training_base.groupBy("event_time_utc", "zone").count().filter("count > 1").show()